<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy python-dotenv

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
daily_files = [
    hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
    for m in MONTHS
]
dim_content_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                    filename="dim_content.parquet", token=token)
clients_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                filename="dim_clients.parquet", token=token)

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in daily_files)
REL = f"read_parquet([{file_list}])"
DECISION_DATE = "2026-03-31"

query = f"""
WITH prior AS (
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions, SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) AS gsc_sum_position,
        SUM(gsc_sum_position) FILTER (WHERE gsc_avg_position > 0) AS pos_weighted_sum,
        SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0) AS pos_weighted_impressions,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        COUNT(*) FILTER (WHERE ga4_sessions > 0) AS days_with_sessions,
        BOOL_OR(ga4_data_available) AS ga4_data_available,
        SUM(ga4_pageviews) AS ga4_pageviews, SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_users) AS ga4_users, SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic, SUM(sessions_direct) AS sessions_direct,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 60 DAY
              AND report_date < DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_baseline_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_recent_impr
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
    GROUP BY content_hash_id
),
future AS (
    SELECT content_hash_id, SUM(gsc_impressions) AS future_impressions
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}'
      AND report_date <= DATE '{DECISION_DATE}' + INTERVAL 30 DAY
    GROUP BY content_hash_id
)
SELECT p.*, f.future_impressions
FROM prior p JOIN future f USING (content_hash_id)
WHERE p.trend_baseline_impr > 0 AND p.trend_recent_impr > 0
"""
df = con.sql(query).df()

df["gsc_avg_position"] = df["pos_weighted_sum"] / df["pos_weighted_impressions"].replace(0, np.nan)

# dim_content carries its own client_hash_id; dropping it avoids a column
# collision that silently breaks the dim_clients merge below.
dim = con.sql(f"SELECT * FROM read_parquet('{dim_content_file}')").df()
dim = dim.drop(columns=["client_hash_id"])
df = df.merge(dim, on="content_hash_id", how="left")

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM read_parquet('{clients_file}')").df()
df = df.merge(clients, on="client_hash_id", how="left")

print("Rows before any filtering:", len(df))

prior_window_start = pd.Timestamp(DECISION_DATE) - pd.Timedelta(days=90)
coverage_ok = df["gsc_data_start"].isna() | (df["gsc_data_start"] <= prior_window_start)
print("Dropped for incomplete client coverage:", (~coverage_ok).sum(),
      f"({(~coverage_ok).mean():.1%})")
df = df[coverage_ok].copy()
print("Rows after coverage filter:", len(df))
print()

df["prior_trend_pct"] = (df["trend_recent_impr"] - df["trend_baseline_impr"]) / df["trend_baseline_impr"] * 100
df["was_declining"] = df["prior_trend_pct"] <= -20

decision_ts = pd.Timestamp(DECISION_DATE)
df["content_age_days"] = (decision_ts - pd.to_datetime(df["content_created_date"])).dt.days
df["days_since_last_update"] = (decision_ts - pd.to_datetime(df["content_updated_date"])).dt.days

df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan) * 100).fillna(0)
df["engagement_rate"] = (df["ga4_engaged_sessions"] / df["ga4_sessions"].replace(0, np.nan) * 100).fillna(0)
df["scroll_rate"] = (df["scroll_events"] / df["ga4_pageviews"].replace(0, np.nan) * 100).fillna(0)

# Flags must be computed BEFORE the fills below, or the missingness is erased.
df["has_position_data"] = df["gsc_avg_position"].notna().astype(int)
df["has_ga4_data"] = df["ga4_data_available"].fillna(False).astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_backlink_data"] = df["backlinks"].notna().astype(int)

df["gsc_avg_position"] = df["gsc_avg_position"].fillna(df["gsc_avg_position"].median())
for col in ["search_volume", "competition", "cpc", "word_count", "char_count", "backlinks",
            "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions",
            "ga4_total_engagement_sec", "sessions_organic", "sessions_direct", "scroll_events"]:
    df[col] = df[col].fillna(0)
df["main_intent"] = df["main_intent"].fillna("unknown")
df["content_type"] = df["content_type"].fillna("unknown")
df["competition_level"] = df["competition_level"].fillna("unknown")

for col in ["gsc_impressions", "gsc_clicks", "ga4_sessions", "search_volume", "backlinks",
            "scroll_events", "gsc_sum_position", "ga4_engaged_sessions"]:
    df[f"log_{col}"] = np.log1p(df[col])

print("Feature vector shape:", df.shape)
df.head()


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows before any filtering: 134398
Dropped for incomplete client coverage: 18652 (13.9%)
Rows after coverage filter: 115746



Feature vector shape: (115746, 67)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,pos_weighted_sum,pos_weighted_impressions,days_with_impressions,days_with_sessions,ga4_data_available,...,has_word_count,has_backlink_data,log_gsc_impressions,log_gsc_clicks,log_ga4_sessions,log_search_volume,log_backlinks,log_scroll_events,log_gsc_sum_position,log_ga4_engaged_sessions
0,content_cfbeca9bc92d1641,client_62f4a7e64f5e0096,427.0,2.0,6369.0,6369.0,387.0,86,0,False,...,0,1,6.059123,1.098612,0.0,2.397895,0.0,0.0,8.759355,0.0
1,content_2ec0bf3e32aef276,client_62f4a7e64f5e0096,669.0,1.0,6065.0,6065.0,647.0,90,0,False,...,1,1,6.507278,0.693147,0.0,2.397895,0.0,0.0,8.710455,0.0
2,content_9c40060f2104e2b5,client_62f4a7e64f5e0096,49182.0,168.0,263989.0,263989.0,49182.0,90,0,False,...,1,1,10.803303,5.129899,0.0,0.0,0.0,0.0,12.483667,0.0
3,content_f26ed43eddff9c03,client_62f4a7e64f5e0096,7087.0,22.0,23743.0,23743.0,7087.0,90,0,False,...,1,1,8.866158,3.135494,0.0,0.0,0.0,0.0,10.075085,0.0
4,content_e7e0d2dddfe26943,client_62f4a7e64f5e0096,2905.0,4.0,8572.0,8572.0,2856.0,90,0,False,...,1,1,7.974533,1.609438,0.0,0.0,0.0,0.0,9.056373,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Known before decision point? |
|---|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` | 90-day GSC totals | zero-fill | Yes |
| `gsc_avg_position` | Impression-weighted 90-day average: `SUM(sum_position)/SUM(impressions)`, both filtered to days with real position data. Not a mean of daily means — that would let one low-traffic day count as much as a high-traffic one. | `has_position_data` flag, then median-fill. Never zero: 0 would read as better than rank 1. | Yes |
| `ctr` | `clicks / impressions × 100` | zero-fill is safe here — 56.8% of tracked pages genuinely have 0 CTR (`w03`) | Yes |
| `days_with_impressions`, `days_with_sessions` | Days of the 90 with any activity — consistency, not volume | not missing (a count) | Yes |
| `content_age_days`, `days_since_last_update` | Days from the decision point back to `content_created_date` / `content_updated_date` | 0% missing | Yes |
| GA4 totals: `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `scroll_events` | 90-day GA4 totals | zero-fill, but only meaningful next to `has_ga4_data` — 51.7% never tracked, 19.5% undetermined (`w03`) | Yes |
| `engagement_rate`, `scroll_rate` | GA4 ratios × 100 | as `ctr` | Yes |
| `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent` | Keyword context (`dim_content`) | `has_keyword_data` flag, then 0 / `"unknown"` (18.5% missing) | Yes |
| `word_count`, `char_count` | Content size | `has_word_count` flag, then zero-fill (30.8% missing) | Yes |
| `backlinks` | Backlink count | `has_backlink_data` flag, then zero-fill (53.0% missing) | Yes |
| `content_type`, `category_count` | Content metadata | `"unknown"` fill; `category_count` 0% missing | Yes |
| `prior_trend_pct`, `was_declining` | 30-vs-30 trend check | not missing by construction | Yes |
| `log_*` (8 columns) | `log1p` of every heavy-tailed count | as the source column | Yes |

**Log, then scale.** `log1p` fixes shape (one page has ~800K impressions); `StandardScaler` fixes scale (raw counts beside 0/1 flags). Scaling happens at fit time on train only, never baked into this frame. Order matters — log needs non-negative input.

**Dropped:** `sessions_ai`, `ai_chatgpt`/`perplexity`/`gemini`, `sessions_referral`/`social`/`paid` — traffic channels unrelated to a GSC-impression label. Sparsity (85-99.7% zero) was the secondary reason. Same logic excludes `ai_traffic_pct`.

**Sum vs. average.** Additive quantities sum. Ratios need numerator and denominator summed first — check what the source columns relate to before choosing.

**Categorical encoding** is an ML-08 decision; this notebook only guarantees clean, correctly-typed strings.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
%pip install -q scikit-learn

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

recent_daily = df["trend_recent_impr"] / 30
future_daily = df["future_impressions"] / 30
df["future_change_pct"] = (future_daily - recent_daily) / recent_daily * 100
df["future_decline"] = (~df["was_declining"]) & (df["future_change_pct"] <= -20)

honest_features = [
    "gsc_avg_position", "log_gsc_sum_position", "prior_trend_pct",
    "log_gsc_impressions", "log_gsc_clicks", "log_ga4_sessions", "log_search_volume", "log_backlinks",
    "log_scroll_events", "log_ga4_engaged_sessions", "word_count", "char_count", "category_count",
    "content_age_days", "days_since_last_update", "ctr", "engagement_rate", "scroll_rate",
    "days_with_impressions", "days_with_sessions",
    "has_position_data", "has_ga4_data", "has_keyword_data", "has_word_count", "has_backlink_data",
]
X = df[honest_features].fillna(0)
y = df["future_decline"].astype(int)
groups = df["client_hash_id"]


def fit_scaled(X_train, y_train, X_test):
    """Scale on train only, then fit. Unscaled input fails to converge here."""
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(max_iter=5000)
    model.fit(scaler.transform(X_train), y_train)
    return model.predict_proba(scaler.transform(X_test))[:, 1]


gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

honest_probs = fit_scaled(X.iloc[train_idx], y.iloc[train_idx], X.iloc[test_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], honest_probs)

print(f"Honest features, grouped split -- test AUC: {honest_auc:.3f}")
print(f"Base rate (future_decline):              {y.mean():.1%}")
print(f"Chance-level AUC:                        0.500")
print(f"Test-set size: {len(test_idx):,} pages across {groups.iloc[test_idx].nunique()} held-out clients")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Honest features, grouped split -- test AUC: 0.426
Base rate (future_decline):              39.1%
Chance-level AUC:                        0.500
Test-set size: 13,078 pages across 6 held-out clients


**AUC 0.426 is below the 0.500 chance level** — across the ranking as a whole this orders held-out clients slightly worse than random. Diagnosing why is signal-audit work; four falsifiable hypotheses for ML-06:

1. **Sign flips across clients.** *Test:* fit per client, compare coefficient signs. The +0.168 random-vs-grouped gap (Attack 3) is consistent with it.
2. **The label carries little page-level signal.** *Test:* correlate `prior_trend_pct` with `future_change_pct`; compare decline rates across prior-trend buckets. A flat rate would mean the target is near-coin-flip by construction.
3. **Cohort selection.** The query keeps only pages with `trend_recent_impr > 0`, which may catch pages at a local peak. *Test:* relax the filter, re-compute the base rate; compare established vs. newly-active pages.
4. **Wrong evaluation scope — possibly a bug, not a model failure.** All numbers rank every held-out page in one pile, forcing scores to compare across clients of 711 to 24,418 pages. *Test:* rank within client, average Precision@K and AUC. If per-client clears 0.5, the scope was the problem. Subsumes #1.

**Population disclosure:** the query inner-joins to the future window, which drops pages absent from it — 1 of 134,399 (0.00%). Negligible, but disclosed per the leakage skill's population-selection rule.

> The top of this same ranking is still enriched — see Precision@K below. A sub-chance AUC alone would have given the wrong conclusion.

**Precision@K.** `w02` §3 named Precision@50 as the governing metric: a specialist works a fixed batch, so "how good is the top 50?" is the real question. `precision_at_k` mirrors `scripts/ml_utils.py`.

In [3]:
def precision_at_k(y_true, scores, k):
    """Mirrors scripts/ml_utils.py; inlined so the notebook is Colab-portable."""
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


y_test = y.iloc[test_idx]
base_rate = y_test.mean()

print(f"Base rate on the held-out clients: {base_rate:.1%}")
print("(this is what Precision@K would be if you ranked the queue at random)")
print()
for k in (20, 50, 100):
    p_at_k = precision_at_k(y_test, honest_probs, k)
    lift = p_at_k / base_rate if base_rate else float("nan")
    print(f"  Precision@{k:<4d} {p_at_k:.3f}   ({p_at_k * k:.0f}/{k} real)   lift vs base rate: {lift:.2f}x")

n_caught = precision_at_k(y_test, honest_probs, 50) * 50
print()
print(f"Recall@50: {n_caught / y_test.sum():.2%} of all {int(y_test.sum()):,} real declines in the test set")

Base rate on the held-out clients: 33.7%
(this is what Precision@K would be if you ranked the queue at random)

  Precision@20   0.600   (12/20 real)   lift vs base rate: 1.78x
  Precision@50   0.580   (29/50 real)   lift vs base rate: 1.72x
  Precision@100  0.540   (54/100 real)   lift vs base rate: 1.60x

Recall@50: 0.66% of all 4,403 real declines in the test set


**The two metrics disagree.**

| Metric | Value | vs. chance |
|---|---|---|
| AUC (whole ranking) | 0.426 | below the 0.500 chance level |
| Precision@50 (top of queue) | 0.580 | 1.72x the 33.7% base rate |

AUC scores the entire ranking; Precision@50 scores only the top 50 of 13,078 pages. The ranking is inverted on average while its top 0.4% is genuinely enriched — 29 real declines where chance gives ~17, roughly 3.6 standard deviations above the base rate.

Precision@K governs, per `w01`: a specialist works a fixed batch, not the whole list.

**Caveats**

1. **Recall@50 = 0.66%** (29 of 4,403 declines). The ceiling at this scope is 1.14% even for a perfect model — a capacity limit, not a model failure.
2. **The scope is probably wrong.** Both figures assume one global queue; per-client at the same K=50 raises the recall ceiling from 0.09% to 19.1% (`w02` §3). ML-06 hypothesis 4.
3. **K=50 is FlyRank convention, but the cadence behind it is unverified** — a weekly cycle would also undercut the 30-day label window.
4. **One split, one decision point, six held-out clients.** A 50-page slice is high variance.

**Feature-importance sanity check.** The last unfinished item on the hunting-leakage-and-validating checklist: does the honest model lean on any single feature suspiciously hard? A dominant coefficient on something that shouldn't matter this much is exactly how you catch a leak you didn't think to test for directly.

In [4]:
# fit_scaled returns only predictions, so refit here to keep the model object.
scaler_check = StandardScaler().fit(X.iloc[train_idx])
model_check = LogisticRegression(max_iter=5000).fit(scaler_check.transform(X.iloc[train_idx]), y.iloc[train_idx])

coefs = pd.Series(model_check.coef_[0], index=honest_features).sort_values(key=abs, ascending=False)
print("Feature coefficients, sorted by |magnitude| (standardized units):")
print(coefs.round(3))

Feature coefficients, sorted by |magnitude| (standardized units):
char_count                 -1.416
word_count                  1.232
log_gsc_impressions         1.015
log_gsc_sum_position       -0.685
log_gsc_clicks             -0.487
gsc_avg_position            0.223
has_keyword_data           -0.200
log_scroll_events           0.145
has_word_count              0.140
has_ga4_data                0.125
days_since_last_update     -0.101
content_age_days            0.076
log_ga4_sessions           -0.073
has_backlink_data           0.070
category_count              0.054
days_with_impressions       0.054
prior_trend_pct             0.053
days_with_sessions          0.047
log_backlinks               0.037
has_position_data           0.022
ctr                         0.019
log_search_volume           0.017
scroll_rate                 0.015
engagement_rate             0.007
log_ga4_engaged_sessions    0.003
dtype: float64


**No leak — multicollinearity.** `char_count` (−1.42) and `word_count` (+1.23) dominate with opposite signs; they correlate at **0.934**. That is two redundant features splitting one signal, not a hidden leak — neither is future-derived. Nothing else is close. For ML-08: trees handle this fine, but a linear model would want one of the two dropped, or their ratio.

**Attack 1: inject the actual label-generating quantity.** `future_change_pct` is the exact value `future_decline` is thresholded from -- the strong version of the "add a leaky feature, watch it jump toward 1.0" test from the hunting-leakage-and-validating skill.

In [5]:
X_leaky1 = X.copy()
X_leaky1["future_change_pct"] = df["future_change_pct"].values
leaky1_probs = fit_scaled(X_leaky1.iloc[train_idx], y.iloc[train_idx], X_leaky1.iloc[test_idx])
leaky1_auc = roc_auc_score(y.iloc[test_idx], leaky1_probs)

print(f"WITH future_change_pct injected -- test AUC: {leaky1_auc:.3f}")
print(f"  jump from honest baseline: {leaky1_auc - honest_auc:+.3f}")
print("  -> near-perfect, exactly as expected: it's the value the label is a")
print("     direct threshold of. This is what a real leak looks like.")

WITH future_change_pct injected -- test AUC: 0.922
  jump from honest baseline: +0.496
  -> near-perfect, exactly as expected: it's the value the label is a
     direct threshold of. This is what a real leak looks like.


**Attack 2: a weaker, indirect leak.** `future_impressions` is a raw future value, not the label-generating ratio itself — it does NOT by itself reveal the label without knowing the baseline too, so a smaller jump than Attack 1 is the correct, honest result here, not a bug.

In [6]:
X_leaky2 = X.copy()
X_leaky2["future_impressions"] = df["future_impressions"].values
leaky2_probs = fit_scaled(X_leaky2.iloc[train_idx], y.iloc[train_idx], X_leaky2.iloc[test_idx])
leaky2_auc = roc_auc_score(y.iloc[test_idx], leaky2_probs)

print(f"WITH future_impressions injected -- test AUC: {leaky2_auc:.3f}")
print(f"  jump from honest baseline: {leaky2_auc - honest_auc:+.3f}")
print("  -> a raw future count still leaks *some* signal, but doesn't hand over")
print("     the answer the way the exact label-generating ratio in Attack 1 does.")

WITH future_impressions injected -- test AUC: 0.510
  jump from honest baseline: +0.084
  -> a raw future count still leaks *some* signal, but doesn't hand over
     the answer the way the exact label-generating ratio in Attack 1 does.


**Attack 3: random split vs. grouped split, honest features only.** Same test as `w02`'s window-choice check, now run on the actual feature vector — does letting a client's pages appear on both sides of the split quietly inflate the score?

In [7]:
train_idx_r, test_idx_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
random_probs = fit_scaled(X.iloc[train_idx_r], y.iloc[train_idx_r], X.iloc[test_idx_r])
random_auc = roc_auc_score(y.iloc[test_idx_r], random_probs)

print(f"Honest features, RANDOM split  -- test AUC: {random_auc:.3f}")
print(f"Honest features, GROUPED split -- test AUC: {honest_auc:.3f}")
print(f"  gap: {random_auc - honest_auc:+.3f} -- the random split's client leakage inflates the score")

Honest features, RANDOM split  -- test AUC: 0.596
Honest features, GROUPED split -- test AUC: 0.426
  gap: +0.170 -- the random split's client leakage inflates the score


**Timeline check.** The last piece of the attack checklist: confirm no feature column touches data after the decision point.

In [8]:
print("Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause")
print("in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.")

Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause
in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded | Why |
|---|---|
| `future_change_pct`, `future_decline`, `future_recovery`, `future_momentum`, `future_impressions` | The label, or its window. Attack 1: injecting `future_change_pct` takes AUC to 0.922. |
| `sessions_ai`, `ai_chatgpt`/`perplexity`/`gemini`, `sessions_referral`/`social`/`paid` | Traffic channels unrelated to a GSC-impression label; also 85-99.7% zero. |
| all of `fact_content_query_90d` | Its window (2026-04-02 → 2026-06-30) overlaps the label window (`w03`). |
| `last_optimized_date`, `optimization_eligible_date` | 87.8% missing; naming and sparsity suggest they populate only when FlyRank acted — product-decision-as-feature. Unverified, so excluded. |
| `provider_used`, `model_used` | Marked "not a model feature" in the data dictionary. |
| `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero in this window — no variance. |
| hash IDs, `report_date`, `month` | Grouping and windowing only. |
| `is_published`, `is_deleted` | Row filters, not signals. |
| Product flags (`health_score`, `priority_score`, `action_type`) | Not shipped in this data. |
| Clients with incomplete prior-window coverage | 18,652 rows (13.9%) dropped — their `gsc_data_start` falls inside the 90-day window. |

*(`w03` lists several of these as candidates — that is the data contract describing columns; this is a modelling decision, not a contradiction.)*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

